<a href="https://colab.research.google.com/github/boyerdan2000-hipr/WarpX-PIC-Beam-Plasma-Wakefield-Propulsion/blob/main/hipr_picmi_dynamic_gradient_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## ==============================================================================
# Cell 1: Download, install, and compile the WarpX PIC environment
# ==============================================================================
!pip uninstall -y pywarpx amrex
!rm -rf WarpX
!git clone https://github.com/ECP-WarpX/WarpX.git
!cmake -S WarpX -B WarpX/build -DWarpX_DIMS=RZ -DWarpX_COMPUTE=CUDA -DWarpX_PYTHON=ON
!cmake --build WarpX/build -j 4 --target pip_install

In [ ]:
# ==============================================================================
# Cell 2: WARPX PICMI, Extended 80-Packet Run (320,000 Steps)
# ==============================================================================
%matplotlib inline
import os
import sys
import numpy as np
import time
from datetime import datetime
import pytz

# 1. MOUNT GOOGLE DRIVE (The Fail-Safe)
from google.colab import drive
drive.mount('/content/drive')
save_dir = '/content/drive/My Drive/WarpX_Checkpoints/'
os.makedirs(save_dir, exist_ok=True)
stats_file = os.path.join(save_dir, 'raw_stats.txt')

with open(stats_file, 'w') as f:
    f.write("step,max_ez,min_ez,avg_abs_ez\n")

# --- THE BULLETPROOF BYPASS ---
sys.path.insert(0, '/content/WarpX/build/lib')
sys.path.insert(0, '/content/WarpX/build/bin')
# ------------------------------

from pywarpx import picmi, fields, warpx

# ==============================================================================
# 1. PHYSICS SETUP & DOMAIN BOUNDARIES
# ==============================================================================
c = 299792458.0
v_b = 0.2 * c
gamma = 1.02062
u_z = v_b * gamma

plasma_density = 1e19
m_p = 1.67262192e-27
e_charge = 1.602176634e-19

z_min, z_max, r_max = -30.0e-3, 2.0e-3, 1.5e-3

grid = picmi.CylindricalGrid(
    number_of_cells=[256, 2048],
    lower_bound=[0, z_min],
    upper_bound=[r_max, z_max],
    lower_boundary_conditions=['none', 'dirichlet'],
    upper_boundary_conditions=['dirichlet', 'dirichlet'],
    lower_boundary_conditions_particles=['none', 'absorbing'],
    upper_boundary_conditions_particles=['absorbing', 'absorbing'],
    n_azimuthal_modes=1
)

solver = picmi.ElectromagneticSolver(grid=grid, method='Yee', cfl=0.99)
plasma_layout = picmi.GriddedLayout(n_macroparticle_per_cell=[2, 2, 1], grid=grid)

plasma_distribution = picmi.UniformDistribution(
    density=plasma_density,
    lower_bound=[0.0, 0.0, 0.0],
    upper_bound=[r_max, 2 * np.pi, 100.0]
)

electrons = picmi.Species(particle_type='electron', name='electrons', initial_distribution=plasma_distribution)
lithium_ions = picmi.Species(particle_type='Li', charge_state=1, mass=7.0*m_p, name='lithium_ions', initial_distribution=plasma_distribution)

# ==============================================================================
# 2. THE 1 AMP SAWTOOTH TRAIN (500 MICRON RADIUS)
# ==============================================================================
beam_radius = 500e-6
packet_length = 1.055e-3
period = 2.11e-3

I_peak = 4.0
beam_area = np.pi * (beam_radius**2)
n0 = I_peak / (e_charge * beam_area * v_b)
print(f"Calculated peak beam density (n0) for 500 microns: {n0:.2e} m^-3")

sawtooth_train = f"{n0} * (x < {beam_radius}) * (z > -21.1e-3) * (z <= 0.0) * " \
                 f"((-z - {period}*floor(-z/{period})) < {packet_length}) * " \
                 f"((-z - {period}*floor(-z/{period})) / {packet_length})"

gold_distribution = picmi.AnalyticDistribution(
    density_expression=sawtooth_train,
    momentum_expressions=['0', '0', f"{u_z}"]
)

gold_beam = picmi.Species(particle_type='Au', charge_state=51, mass=197.0*m_p, name='gold_beam', initial_distribution=gold_distribution)

sim = picmi.Simulation(solver=solver, max_steps=320000, verbose=0)
sim.add_species(electrons, layout=plasma_layout)
sim.add_species(lithium_ions, layout=plasma_layout)
sim.add_species(gold_beam, layout=picmi.GriddedLayout(n_macroparticle_per_cell=[2, 4, 1], grid=grid))

# ==============================================================================
# 3. DIRECT AMREX C++ OVERRIDES & CHECKPOINTED EXECUTION LOOP
# ==============================================================================
warpx.moving_window_dir = 'z'
warpx.moving_window_v = v_b

tz_pdt = pytz.timezone('America/Los_Angeles')
start_time = datetime.now(tz_pdt)
print(f"Simulation started at: {start_time.strftime('%Y-%m-%d %H:%M:%S PDT')}")
print("Running 320,000 steps with Drive Checkpointing...")

window_size = 10
kernel = np.ones(window_size) / window_size

loop_start_time = time.time()

for chunk in range(80):
    chunk_start_time = time.time()

    sim.step(4000)
    step_count = (chunk + 1) * 4000

    # 1. Extract from VRAM
    ez_grid = sim.fields.get('E', 'z')[...]
    if ez_grid.shape[0] > ez_grid.shape[1]:
        ez_on_axis = ez_grid[:, 0]
    else:
        ez_on_axis = ez_grid[0, :]

    Ez_MVm_current_chunk = ez_on_axis / 1e6

    # 2. Extract and save exact raw peaks to text file
    chunk_max = np.max(Ez_MVm_current_chunk)
    chunk_min = np.min(Ez_MVm_current_chunk)
    chunk_avg_abs = np.mean(np.abs(Ez_MVm_current_chunk))

    with open(stats_file, 'a') as f:
        f.write(f"{step_count},{chunk_max},{chunk_min},{chunk_avg_abs}\n")

    # 3. Compress: Moving average + 10x spatial stride + 32-bit cast
    local_moving_avg = np.convolve(Ez_MVm_current_chunk, kernel, mode='same')
    reduced_chunk = local_moving_avg[::10].astype(np.float32)

    # 4. Save safely to Drive
    chunk_filename = f"{save_dir}ez_chunk_{step_count}.npy"
    np.save(chunk_filename, reduced_chunk)

    # 5. Flush AMReX memory completely
    del ez_grid
    del ez_on_axis
    del Ez_MVm_current_chunk
    del local_moving_avg
    del reduced_chunk

    # 6. Timing and Heartbeat
    chunk_elapsed = time.time() - chunk_start_time
    total_elapsed = time.time() - loop_start_time
    current_time_str = datetime.now(tz_pdt).strftime('%H:%M:%S PDT')

    print(f"[{current_time_str}] --> Au51+ Packet {chunk + 1}/80 injected. Step {step_count} written to Drive. (Chunk time: {chunk_elapsed/60:.1f} min)")

In [ ]:
# ==============================================================================
# Cell 3: Data Analysis and Plotting
# ==============================================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from google.colab import drive

# 1. Mount Drive (Required if running after a session disconnect)
drive.mount('/content/drive')
save_dir = '/content/drive/My Drive/WarpX_Checkpoints/'

print("Scanning Google Drive for saved checkpoints...")

# 2. Load exact raw AC statistics
stats_file = os.path.join(save_dir, 'raw_stats.txt')
if os.path.exists(stats_file):
    stats_df = pd.read_csv(stats_file)
    global_max_ez = stats_df['max_ez'].max()
    global_min_ez = stats_df['min_ez'].min()
    global_avg_abs_ez = stats_df['avg_abs_ez'].mean()
else:
    print("Warning: raw_stats.txt not found. Metrics will be skipped.")

# 3. Locate, sort, and stitch the compressed array chunks
chunk_files = glob.glob(os.path.join(save_dir, 'ez_chunk_*.npy'))

if not chunk_files:
    raise FileNotFoundError("No chunk files found in Drive. Ensure Cell 2 has saved data.")

# Sort numerically by step count to ensure perfect chronological order
chunk_files.sort(key=lambda x: int(x.split('_chunk_')[-1].split('.npy')[0]))

print(f"Successfully loaded {len(chunk_files)} snapshot chunks.")
compressed_snapshots = [np.load(f) for f in chunk_files]

# Use the last available snapshot for the 1D steady-state plot
final_snapshot = compressed_snapshots[-1]
final_step = int(chunk_files[-1].split('_chunk_')[-1].split('.npy')[0])

# 4. Reconstruct the spatial axis based on the original 2048 cells and 10x stride
z_min, z_max = -30.0e-3, 2.0e-3
original_num_cells = 2048
original_dz = (z_max - z_min) / original_num_cells

stride = 10
new_dz = original_dz * stride
z_array_compressed = np.linspace(z_min, z_max, len(final_snapshot)) * 1e3 # mm

# 5. Apply Butterworth low-pass filter to the final snapshot
fs = 1.0 / new_dz
cutoff = 0.1 / original_dz
nyquist = 0.5 * fs
b, a = butter(4, cutoff / nyquist, btype='low', analog=False)
Ez_dc_filtered = filtfilt(b, a, final_snapshot)

# ==============================================================================
# PLOT 1: 1D Macroscopic Ambipolar Towing Field (Final Available Step)
# ==============================================================================
plt.figure(figsize=(12, 6))
plt.plot(z_array_compressed, final_snapshot, label='Downsampled Wakefield ($E_z$)', color='blue', alpha=0.25)
plt.plot(z_array_compressed, Ez_dc_filtered, label='Macroscopic DC Ambipolar Field', color='red', linewidth=3)
plt.axhline(0, color='black', linewidth=1)
plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.title(f'Macroscopic Ponderomotive Acceleration & Ambipolar Towing\nLow-Pass Filtered Longitudinal Wakefield at Step {final_step:,}', fontsize=14, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# ==============================================================================
# PLOT 2: 2D Spatiotemporal Evolution Heatmap
# ==============================================================================
evolution_matrix = np.vstack(compressed_snapshots)

# Dynamically generate the step array based on how many chunks actually loaded
step_array = np.arange(1, len(compressed_snapshots) + 1) * 4000

plt.figure(figsize=(14, 8))
limit = np.percentile(np.abs(evolution_matrix), 95)

pcm = plt.pcolormesh(
    z_array_compressed,
    step_array,
    evolution_matrix,
    cmap='RdBu_r',
    vmin=-limit,
    vmax=limit,
    shading='auto'
)

cbar = plt.colorbar(pcm, pad=0.02)
cbar.set_label('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Simulation Step', fontsize=12)
plt.title(f'Spatiotemporal Evolution of the Longitudinal Wakefield\nContinuous Au51+ Train Injection (0 to {final_step:,} Steps)', fontsize=14, fontweight='bold')
plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.ylim(np.min(step_array), np.max(step_array))
plt.grid(True, linestyle='--', color='black', alpha=0.2)
plt.tight_layout()
plt.show()

# ==============================================================================
# PRINT FINAL METRICS
# ==============================================================================
if os.path.exists(stats_file):
    print("==========================================================")
    print("   TRUE RAW AC WAKEFIELD GRADIENT STATISTICS")
    print("==========================================================")
    print(f"Maximum Accelerating Gradient :  {global_max_ez:.2f} MV/m")
    print(f"Maximum Decelerating Gradient : {global_min_ez:.2f} MV/m")
    print(f"Average Absolute Gradient     :   {global_avg_abs_ez:.2f} MV/m")
    print("==========================================================")

print("   MACROSCOPIC AMBIPOLAR TOWING FIELD (DC FILTERED)")
print("==========================================================")
print(f"Max DC Accelerating Gradient  :  {np.max(Ez_dc_filtered):.2f} MV/m")
print(f"Max DC Decelerating Gradient  : {np.min(Ez_dc_filtered):.2f} MV/m")
print(f"Average Net Forward Pressure  :   {np.mean(Ez_dc_filtered):.2f} MV/m")
print("==========================================================")

In [ ]:
# ==============================================================================
# Cell 4: Wakefield spatiotemporal heatmap
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt

# Verify that the variables exist from Cell 3
if 'compressed_snapshots' not in locals() or 'z_array_compressed' not in locals():
    raise NameError("Data not found in RAM. Please ensure Cell 3 successfully ran first.")

# 1. Stack the individual 1D array snapshots into a single 2D matrix
evolution_matrix = np.vstack(compressed_snapshots)

# 2. Dynamically generate the step array based on the number of loaded chunks
step_array = np.arange(1, len(compressed_snapshots) + 1) * 4000
final_step_loaded = step_array[-1]

# 3. Create the 2D Color Map
plt.figure(figsize=(14, 8))

# Calculate a symmetrical color limit based on the 95th percentile
# This prevents extreme localized spikes from washing out the color contrast
limit = np.percentile(np.abs(evolution_matrix), 95)

# Plot the 2D matrix (Red = Accelerating/Positive, Blue = Decelerating/Negative)
pcm = plt.pcolormesh(
    z_array_compressed,
    step_array,
    evolution_matrix,
    cmap='RdBu_r',
    vmin=-limit,
    vmax=limit,
    shading='auto'
)

# 4. Formatting and Labels
cbar = plt.colorbar(pcm, pad=0.02)
cbar.set_label('Electric Field, $E_z$ (MV/m)', fontsize=12)

plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Simulation Step', fontsize=12)
plt.title(f'Spatiotemporal Evolution of the Longitudinal Wakefield\nContinuous Au51+ Train Injection (0 to {final_step_loaded:,} Steps)', fontsize=14, fontweight='bold')

plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.ylim(np.min(step_array), np.max(step_array))

# Add a subtle grid to track packet injections
plt.grid(True, linestyle='--', color='black', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# Cell 5: Longitudinal Phase Space Map - run only after complete cell 2 run
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pywarpx import warpx

print("Attempting to extract Lithium Ion phase space data from the active AMReX backend...")

try:
    # 1. Extract raw arrays directly from VRAM
    # In WarpX, 'uz' represents the momentum per unit mass (gamma * v_z)
    z_li_raw = warpx.get_particle_z('lithium_ions')
    uz_li_raw = warpx.get_particle_uz('lithium_ions')
except Exception as e:
    print("\n[!] CRITICAL ERROR: Particle data not found in VRAM.")
    print("[!] Cell 5 cannot be run from Drive checkpoints. It must be executed immediately after Cell 2 completes in an active Colab session.")
    raise e

num_particles = len(z_li_raw)
print(f"Total Lithium Macroparticles found: {num_particles:,}")

# 2. Subsample to prevent frontend memory crashes
sample_size = min(num_particles, 500000)
print(f"Subsampling {sample_size:,} particles for 2D histogram visualization...")

# Randomly select indices to get a representative distribution without bias
indices = np.random.choice(num_particles, sample_size, replace=False)

# Convert z to millimeters
z_sample_mm = z_li_raw[indices] * 1e3

# Convert uz to standard dimensionless momentum representation (gamma * beta_z)
c = 299792458.0
pz_dimensionless = uz_li_raw[indices] / c

# 3. Create the 2D Phase Space Density Plot
plt.figure(figsize=(12, 7))

# hist2d maps particle density to color
h, xedges, yedges, image = plt.hist2d(
    z_sample_mm,
    pz_dimensionless,
    bins=[250, 150],
    cmap='inferno',
    cmin=1  # Only color bins with at least 1 particle
)

# 4. Formatting and Labels
cbar = plt.colorbar(image, pad=0.02)
cbar.set_label('Macroparticle Count Density', fontsize=12)

plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Dimensionless Momentum, $p_z / m_{Li} c$', fontsize=12)
plt.title('Lithium Ion Longitudinal Phase Space\nBulk Momentum Transfer at Final Step', fontsize=14, fontweight='bold')

# Reference line for zero momentum
plt.axhline(0, color='white', linestyle='--', alpha=0.5, linewidth=1.5)

plt.xlim(np.min(z_sample_mm), np.max(z_sample_mm))
plt.tight_layout()
plt.show()

# 5. Flush VRAM
del z_li_raw
del uz_li_raw
del z_sample_mm
del pz_dimensionless
print("Phase space extraction complete. VRAM cleared.")